In [ ]:
import pandas as pd

df = pd.read_csv("../ml/data/raw/data.csv")
print(df.shape)
df.head()

(7691, 10)


<bound method NDFrame.head of                                              house_type          locality  \
0     2 BHK Flat for Rent in Oberoi Woods, Goregaon ...     Goregaon East   
1     1 BHK Flat for Rent in Sapphire Lakeside, Powa...             Powai   
2                  1 BHK House for Rent in Mundhwa Pune           Mundhwa   
3                 2 BHK Flat for Rent in Hingna, Nagpur            Hingna   
4     1 BHK Flat for Rent in Unique Star Harsh Vihar...         Mira Road   
...                                                 ...               ...   
7686            3 BHK Flat for Rent in Saket, New Delhi             Saket   
7687  2 BHK Flat for Rent in Phase 7 JP Nagar, Banga...  Phase 7 JP Nagar   
7688  2 BHK Flat for Rent in Malpani Vivanta, Balewa...          Balewadi   
7689    1 BHK Flat for Rent in Shalimar bagh, New Delhi     Shalimar bagh   
7690  2 BHK Flat for Rent in Govindpura, Dooravani N...        Govindpura   

           city    area  beds  bathrooms  bal

In [3]:
df.columns

Index(['house_type', 'locality', 'city', 'area', 'beds', 'bathrooms',
       'balconies', 'furnishing', 'area_rate', 'rent'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7691 entries, 0 to 7690
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   house_type  7691 non-null   str    
 1   locality    7691 non-null   str    
 2   city        7691 non-null   str    
 3   area        7691 non-null   float64
 4   beds        7691 non-null   int64  
 5   bathrooms   7691 non-null   int64  
 6   balconies   7691 non-null   int64  
 7   furnishing  7691 non-null   str    
 8   area_rate   7691 non-null   float64
 9   rent        7691 non-null   float64
dtypes: float64(3), int64(3), str(4)
memory usage: 601.0 KB


In [5]:
df.isnull().sum()

house_type    0
locality      0
city          0
area          0
beds          0
bathrooms     0
balconies     0
furnishing    0
area_rate     0
rent          0
dtype: int64

In [6]:
check = df[["area", "area_rate", "rent"]].copy()

In [7]:
check = df[["area", "area_rate", "rent"]].copy()
check["rent_div_area"] = check["rent"] / check["area"]
check.head(10)

,area,area_rate,rent,rent_div_area
0,897.0,134.000000,120000.0,133.779264
1,490.0,82.000000,40000.0,81.632653
2,550.0,22.000000,12000.0,21.818182
3,1000.0,8.000000,8000.0,8.000000
4,595.0,25.000000,15000.0,25.210084
5,600.0,27.000000,16000.0,26.666667
6,1260.0,47.666667,60000.0,47.619048
7,2700.0,74.111111,200000.0,74.074074
8,650.0,17.000000,11000.0,16.923077
9,495.0,48.000000,24000.0,48.484848


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df["rent"].describe()

count    7.691000e+03
mean     5.479508e+04
std      9.742006e+04
min      1.000000e+03
25%      1.500000e+04
50%      2.800000e+04
75%      5.700000e+04
max      2.700000e+06
Name: rent, dtype: float64

In [10]:
check = df[["area", "area_rate", "rent"]].copy()
check["rent_div_area"] = check["rent"] / check["area"]
check.head(10)

,area,area_rate,rent,rent_div_area
0,897.0,134.000000,120000.0,133.779264
1,490.0,82.000000,40000.0,81.632653
2,550.0,22.000000,12000.0,21.818182
3,1000.0,8.000000,8000.0,8.000000
4,595.0,25.000000,15000.0,25.210084
5,600.0,27.000000,16000.0,26.666667
6,1260.0,47.666667,60000.0,47.619048
7,2700.0,74.111111,200000.0,74.074074
8,650.0,17.000000,11000.0,16.923077
9,495.0,48.000000,24000.0,48.484848


In [11]:
target = "rent"
features = ["locality", "city", "area", "beds", "bathrooms", "furnishing" ]

x = df[features]
y = df[target]

print(x.shape, y.shape)
print(df["locality"].nunique(), df["city"].nunique(), df["furnishing"].nunique())


(7691, 6) (7691,)
1984 5 3


In [12]:
x.dtypes

locality          str
city              str
area          float64
beds            int64
bathrooms       int64
furnishing        str
dtype: object

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [14]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print(x_train.shape, x_test.shape)

(6152, 6) (1539, 6)


In [15]:
cat_cols = ["locality", "city", "furnishing"]
num_cols = ["area", "beds", "bathrooms"]

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

print("preprocessing ready")

preprocessing ready


In [16]:
model = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

print("pipeline ready")

pipeline ready


In [17]:
model.fit(x_train, y_train)

print("training done")

training done


In [18]:
pred = model.predict(x_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("R2 :", r2_score(y_test, pred))

MAE: 18389.26729410976
R2 : 0.70479358353345


In [19]:
print(y.describe())
print("Median rent:", y.median())
print("95th percentile:", y.quantile(0.95))
print("99th percentile:", y.quantile(0.99))

count    7.691000e+03
mean     5.479508e+04
std      9.742006e+04
min      1.000000e+03
25%      1.500000e+04
50%      2.800000e+04
75%      5.700000e+04
max      2.700000e+06
Name: rent, dtype: float64
Median rent: 28000.0
95th percentile: 180000.0
99th percentile: 430000.0


In [20]:
import numpy as np
y_train_log = np.log1p(y_train)
model.fit(x_train, y_train_log)

pred_log = model.predict(x_test)
pred = np.expm1(pred_log)

print("MAE", mean_absolute_error(y_test, pred))
print("R2:", r2_score(y_test, pred))

MAE 18549.95108904285
R2: 0.7016955510023041


In [21]:
model.fit(x_train, y_train)


names = model.named_steps["prep"].get_feature_names_out()
imp = model.named_steps["rf"].feature_importances_

import pandas as pd
imp_df = pd.Series(imp, index=names).sort_values(ascending=False)
print(imp_df.head(10))

num__bathrooms                   0.299103
num__area                        0.294629
cat__city_Mumbai                 0.143143
cat__locality_Worli              0.028365
num__beds                        0.021954
cat__locality_Golf Links         0.017094
cat__furnishing_Furnished        0.015134
cat__locality_Jogeshwari East    0.013232
cat__locality_Prabhadevi         0.012575
cat__locality_Sat Bari           0.010783
dtype: float64


In [22]:
counts = df["locality"].value_counts()

print("Total localities:", len(counts))
print("Localities with 1 row only:", (counts == 1).sum())
print("Localities with 10+ rows:", (counts >= 10).sum())
print("Rows covered by top 50:", counts.head(50).sum(), "of", len(df))

Total localities: 1984
Localities with 1 row only: 1004
Localities with 10+ rows: 172
Rows covered by top 50: 2019 of 7691
